# 🎮 Multi-Objective PPO (MO-PPO) for Super Mario Bros (`mo-supermario-v0`)
### 🏆 Group 1 - Đồ án Tốt nghiệp / Cuối kỳ Học Tăng Cường (Reinforcement Learning)

Notebook này được thiết kế tối ưu để chạy trên **Google Colab (GPU T4 miễn phí)**, kết nối trực tiếp với **Google Drive** để tự động lưu Checkpoint và khôi phục (Resume) xuyên suốt các phiên làm việc 12 giờ.

#### 📌 Mục tiêu:
1. **Mục tiêu 8 điểm:** Mario vượt qua các chướng ngại vật và chạm cờ (reaches the flag) tại màn 1-1 ổn định (Win Rate >= 90%).
2. **Mục tiêu 2 điểm Đua Top (Ranking):** Tối ưu hóa thời gian Speedrun hoặc ăn điểm số tối đa bằng cách chuyển đổi vector trọng số $\mathbf{w}$.

## 1. Kiểm tra phần cứng GPU

In [ ]:
!nvidia-smi

## 2. Kết nối Google Drive để lưu Checkpoints vĩnh viễn
Checkpoint sẽ được lưu tự động tại `/content/drive/MyDrive/DRL_Mario_Checkpoints/` để không bị mất khi hết phiên làm việc.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository từ GitHub hoặc thiết lập mã nguồn

In [ ]:
# Thay đổi đường dẫn repository của nhóm
!git clone https://github.com/TTins1702/DRL-Mo-Supermario-group-1-test.git
%cd DRL-Mo-Supermario-group-1-test

## 4. Cài đặt môi trường & Khóa phiên bản thư viện
> **LƯU Ý CỰC KỲ QUAN TRỌNG:** Phải cài `numpy==1.26.4` (numpy < 2.0) để tương thích với `nes-py` và `gym-super-mario-bros`.

In [ ]:
!bash scripts/setup_env.sh

## 5. Chạy Kiểm thử Toàn bộ Hệ thống (PyTest Suite)
Xác nhận cả 4 phân hệ (Env, Model, Agent, Utils) hoạt động hoàn hảo trước khi huấn luyện.

In [ ]:
!pytest -v tests/

## 6. Khởi động TensorBoard trực tiếp trong Notebook

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

## 7. Bắt đầu Huấn luyện Baseline 8 Điểm (Màn 1-1 Clear)

In [ ]:
# Huấn luyện 2,000,000 timesteps với 8 môi trường song song
!python train.py --config configs/base_8pt_clear.yaml

## 8. Khôi phục Huấn luyện (Resume) nếu phiên bị ngắt kết nối
`CheckpointManager` sẽ tự động tìm thấy checkpoint `latest.pt` mới nhất trong Google Drive và tiếp tục từ đúng update/step trước đó.

In [ ]:
!python train.py --config configs/base_8pt_clear.yaml --resume auto

## 9. Đánh giá Mô hình & Xuất Video MP4 (20 Episodes)

In [ ]:
!python evaluate.py \
  --checkpoint /content/drive/MyDrive/DRL_Mario_Checkpoints/base_8pt_clear/best_winrate.pt \
  --episodes 20 \
  --video-dir ./eval_videos

## 10. Xem Video Gameplay Mario Trực tiếp trong Colab

In [ ]:
import glob
import os
from IPython.display import HTML, display
from base64 import b64encode

videos = sorted(glob.glob("./eval_videos/*.mp4"))
if videos:
    target_video = videos[0]
    print(f"Đang phát video: {target_video}")
    mp4 = open(target_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("Chưa có video được xuất.")

## 11. Fine-tune Đua Top 2 Điểm (Speedrun / High Score)
Tận dụng mạng đã học cách qua màn để tinh chỉnh theo mục tiêu tốc độ cao nhất:

In [ ]:
!python train.py \
  --config configs/bonus_speedrun.yaml \
  --resume /content/drive/MyDrive/DRL_Mario_Checkpoints/base_8pt_clear/best_winrate.pt \
  --reset-lr